In [75]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from churn_prediction.paths import FEATURES_DIR, PROCESSED_DIR, MODELS_OLIST_DIR 
import pickle
import warnings
warnings.filterwarnings('ignore')

In [76]:
print("\n1. Đọc dữ liệu...")
df = pd.read_parquet(FEATURES_DIR / 'customer_features_labeled.parquet')
print(f"   Dữ liệu shape: {df.shape}")
print(f"   Số khách hàng: {len(df):,}")



1. Đọc dữ liệu...
   Dữ liệu shape: (96136, 50)
   Số khách hàng: 96,136


In [77]:
print(df.columns)

Index(['customer_unique_id', 'recency', 'frequency', 'monetary', 'rfm_segment',
       'avg_delivery_time', 'std_delivery_time', 'max_delivery_time',
       'avg_estimated_delivery', 'avg_delivery_delay', 'std_delivery_delay',
       'max_delivery_delay', 'avg_freight_per_order', 'late_delivery_ratio',
       'avg_review_score', 'std_review_score', 'min_review_score',
       'num_comments', 'num_titles', 'avg_days_to_answer', 'low_review_ratio',
       'num_bad_reviews', 'avg_items_per_order', 'std_items_per_order',
       'weekend_purchase_ratio', 'night_purchase_ratio', 'total_orders',
       'avg_gap', 'std_gap', 'max_gap', 'trend_gap', 'avg_installments',
       'favorite_payment_type', 'credit_card_ratio', 'boleto_ratio',
       'spending_trend', 'order_trend', 'recent_vs_old_ratio',
       'avg_order_trend', 'total_comments_msg', 'total_comments_title',
       'unique_sellers', 'customer_state', 'num_categories_bought',
       'review_norm', 'delivery_norm', 'low_review_norm', 'd

### 3. Loại bỏ các cột không cần thiết

In [88]:
# Define target
target_col = 'experience_level'

drop_cols = [
    'customer_unique_id',
    'experience_score',
    'experience_level',

    'review_norm',
    'delivery_norm',
    'low_review_norm',
    'delay_norm',

    'avg_review_score',
    'std_review_score',
    'min_review_score',
    'low_review_ratio',
    'num_bad_reviews',

    'avg_delivery_delay',
    'std_delivery_delay',
    'max_delivery_delay',
    'late_delivery_ratio'
]

cols_to_drop = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=cols_to_drop)
y = df[target_col]

print(f"Target distribution:\n{y.value_counts()}")
print(f"Number of features after cleaning: {X.shape[1]}")

Target distribution:
experience_level
excellent    71174
good         10223
average      10025
poor          4714
Name: count, dtype: int64
Number of features after cleaning: 34


In [89]:
# 5. Encode target labels if needed
if y.dtype == 'object':
    le_target = LabelEncoder()
    y = le_target.fit_transform(y)
    with open(MODELS_OLIST_DIR / 'experience_label_encoder.pkl', 'wb') as f:
        pickle.dump(le_target, f)
    class_names = le_target.classes_
else:
    class_names = ['average', 'excellent', 'good', 'poor']

### 4. Xử lý missing values

In [90]:
print("\n4. Xử lý missing values...")
missing_before = df.isnull().sum().sum()
df = df.fillna(0)  
print(f"Missing trước: {missing_before}, sau: {df.isnull().sum().sum()}")


4. Xử lý missing values...
Missing trước: 0, sau: 0


### 5. Mã hóa biến phân loại (categorical)

In [91]:
#  Encode remaining categorical columns
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print("Encoding categorical columns:", cat_cols)
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

with open(MODELS_OLIST_DIR / 'categorical_encoders_experience.pkl', 'wb') as f:
    pickle.dump(encoders, f)

Encoding categorical columns: ['rfm_segment', 'favorite_payment_type', 'customer_state']


### 6. Feature Selection (chọn đặc trưng quan trọng)

In [92]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print(f"Train size: {X_train.shape[0]}, Val size: {X_val.shape[0]}, Test size: {X_test.shape[0]}")

Train size: 67295, Val size: 14420, Test size: 14421


In [93]:
# Feature selection chọn các đặc trưng (Mutual Information for multiclass) 
print("Performing feature selection...")
selector = SelectKBest(score_func=mutual_info_classif, k=30)
X_train_selected = selector.fit_transform(X_train, y_train)
X_val_selected = selector.transform(X_val)
X_test_selected = selector.transform(X_test)

selected_features = X_train.columns[selector.get_support()].tolist()
print(f"Selected {len(selected_features)} features:")
print(selected_features)

# Save selector
with open(MODELS_OLIST_DIR / 'feature_selector_experience.pkl', 'wb') as f:
    pickle.dump(selector, f)

Performing feature selection...
Selected 30 features:
['recency', 'frequency', 'monetary', 'rfm_segment', 'avg_delivery_time', 'std_delivery_time', 'max_delivery_time', 'avg_estimated_delivery', 'avg_freight_per_order', 'num_comments', 'num_titles', 'avg_days_to_answer', 'avg_items_per_order', 'night_purchase_ratio', 'total_orders', 'avg_gap', 'max_gap', 'avg_installments', 'favorite_payment_type', 'credit_card_ratio', 'boleto_ratio', 'spending_trend', 'order_trend', 'recent_vs_old_ratio', 'avg_order_trend', 'total_comments_msg', 'total_comments_title', 'unique_sellers', 'customer_state', 'num_categories_bought']


### 8. Chuẩn hóa dữ liệu (StandardScaler)

In [94]:
print("\n8. Chuẩn hóa dữ liệu...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_val_scaled = scaler.transform(X_val_selected)
X_test_scaled = scaler.transform(X_test_selected)

with open(MODELS_OLIST_DIR / 'scaler_experience.pkl', 'wb') as f:
    pickle.dump(scaler, f)


8. Chuẩn hóa dữ liệu...


### 9. Lưu các tập dữ liệu đã xử lý

In [95]:
print("\n9. Lưu dữ liệu đã xử lý")
# Lưu dạng numpy arrays để dễ dùng cho modeling
np.save(MODELS_OLIST_DIR / 'X_train_exp.npy', X_train_scaled)
np.save(MODELS_OLIST_DIR / 'X_val_exp.npy', X_val_scaled)
np.save(MODELS_OLIST_DIR / 'X_test_exp.npy', X_test_scaled)
np.save(MODELS_OLIST_DIR / 'y_train_exp.npy', y_train)
np.save(MODELS_OLIST_DIR / 'y_val_exp.npy', y_val)
np.save(MODELS_OLIST_DIR / 'y_test_exp.npy', y_test)

# Cũng lưu dạng pandas để tiện kiểm tra
pd.DataFrame(X_train_scaled, columns=selected_features).to_parquet(MODELS_OLIST_DIR / 'X_train.parquet')
pd.DataFrame(y_train).to_parquet(MODELS_OLIST_DIR / 'y_train.parquet')


9. Lưu dữ liệu đã xử lý


### 10. Tổng kết pipeline

In [96]:
print("DATA PIPELINE HOÀN TẤT!")

print("\nCác file đã tạo:")
print("- label_encoders.pkl: để mã hóa categorical features khi dự đoán mới")
print("- feature_selector.pkl: để chọn đúng features cho dữ liệu mới")
print("- scaler.pkl: để chuẩn hóa dữ liệu mới")
print("- X_train.npy, X_val.npy, X_test.npy: features đã chuẩn hóa")
print("- y_train.npy, y_val.npy, y_test.npy: labels")
print("\nSẵn sàng cho huấn luyện mô hình và tích hợp multi-agent!")

DATA PIPELINE HOÀN TẤT!

Các file đã tạo:
- label_encoders.pkl: để mã hóa categorical features khi dự đoán mới
- feature_selector.pkl: để chọn đúng features cho dữ liệu mới
- scaler.pkl: để chuẩn hóa dữ liệu mới
- X_train.npy, X_val.npy, X_test.npy: features đã chuẩn hóa
- y_train.npy, y_val.npy, y_test.npy: labels

Sẵn sàng cho huấn luyện mô hình và tích hợp multi-agent!
